# Family F as a level in 2020-23

Is family F already alike inside, in number of wings, propulsor band, wing configuration, tractor/pusher arrangement and booms present, and how does it compare with the designs not in F? Separate from the within-F disparity test (`../06_weight_sensitivity`, test 2), which measures a fall over time.

All computing is in `11A20_ff_level.py`, which reads the dataset through the thesis code and the P1 reliability code (`../../reliability`). **Run All** reads eVTOL_dataset_v1 again. Outputs: `eVTOL_dataset_v1/results/family_F_level/` on the Drive. Results are reported, not interpreted.

In [ ]:
import pandas as pd, ff_level as ff   # ff_level puts ../../reliability on the path
import core
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.float_format', lambda x: f'{x:.4f}')
ff.OUT.mkdir(parents=True, exist_ok=True)
base = core.load()
df = ff.design_frame(base)
df.to_csv(ff.OUT / 'designs.csv', index=False)
df.groupby(['cohort', 'F']).size().unstack().rename(columns={True: 'F', False: 'not F'})

## Tasks 1 and 2: the most common option, F and not F, 2016-19 and 2020-23, both bases
File: `task1_2_top_options.csv`.

In [ ]:
top = ff.top_table(df)
top.to_csv(ff.OUT / 'task1_2_top_options.csv', index=False)
top

## Tasks 3 and 4: F against the designs not in F, 2020-23
Top-option share difference with the Newcombe 95 % interval (built from the two Wilson intervals), permutation test (F label shuffled among the 2020-23 designs of the base, 10 000 shuffles, seed 42), Holm over the five dimensions per base; Rao's Q of each dimension alone with the band of the same shuffles. `F alike as a level` = difference above 0, Newcombe low above 0, above the 97.5 % band edge, and Holm p under 0.05. File: `task3_4_level_tests.csv`.

In [ ]:
lt = ff.level_tests(df)
lt.to_csv(ff.OUT / 'task3_4_level_tests.csv', index=False)
lt.T

### Check: Q against the thesis code
`lib.distance.gower_distance` on the one dimension and `lib.dominant_design.mean_pairwise_distance`; the run stops if they differ. File: `check_q_thesis_code.csv`.

In [ ]:
chk = ff.check_q_against_thesis_code(base, df)
chk.to_csv(ff.OUT / 'check_q_thesis_code.csv', index=False)
chk

## Task 5: coding simulation
15 % per coded value, 1 000 replicas, seed 20261118 (`reliability/perturb.py`, rules approved 2026-11-22). Each replica rebuilds the designs, F and the five dimensions, then reruns tasks 1 and 3. Saved every 50 replicas to `simulation_replicas.jsonl`; a restart continues. Files: `simulation_summary.csv`, `simulation_tests_per_replica.csv`, `simulation_top_per_replica.csv`.

In [ ]:
print('seconds per replica on one core:', round(ff.time_replicas(2), 1), '(about 6 min for 1 000 on 14 workers)')
ff.run_simulation(ff.SIM_N, 14)

In [ ]:
sim = ff.summary_simulation(lt)
sim